In [1]:
%%capture
# 로컬 환경용 설치 (RTX 5060ti, 16GB VRAM)
%pip install "transformers>=4.55.0" "bitsandbytes>=0.43.0"
%pip install "torch>=2.0.0" "accelerate>=0.30.0"

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import time

# GPU 확인
if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu_stats.name}")
    print(f"최대 메모리: {gpu_stats.total_memory / 1024**3:.2f} GB")
else:
    print("GPU를 사용할 수 없습니다. CUDA 환경을 확인하세요.")

GPU: NVIDIA GeForce RTX 5060 Ti
최대 메모리: 15.93 GB


# Quantization

## Post Training Quantization

In [3]:
# 사용할 모델 (경량 모델 선택 - Gated가 아닌 모델)
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# FP16 모델 로딩
print("FP16 모델 로딩 중...")
torch.cuda.reset_peak_memory_stats()  # 메모리 통계 초기화
model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 모델 로딩 직후 메모리 측정 (가중치 메모리)
model_memory_fp16 = torch.cuda.memory_allocated() / 1024**3
print(f"모델 로딩 완료: {model_name}")
print(f"FP16 모델 메모리: {model_memory_fp16:.2f} GB")

FP16 모델 로딩 중...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
Skipping import of cpp extensions due to incompatible torch version 2.7.0+cu128 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

모델 로딩 완료: Qwen/Qwen2.5-1.5B-Instruct
FP16 모델 메모리: 2.88 GB


In [4]:
# 모델 파라미터 수 확인
total_params = sum(p.numel() for p in model_fp16.parameters())
trainable_params = sum(p.numel() for p in model_fp16.parameters() if p.requires_grad)

print("=" * 50)
print("[모델 파라미터 분석]")
print("=" * 50)
print(f"총 파라미터 수: {total_params:,}")
print(f"학습 가능한 파라미터: {trainable_params:,}")

# 메모리 계산: 파라미터 수 × 바이트
memory_fp32 = total_params * 4 / 1024**3  # 32비트 = 4바이트
memory_fp16 = total_params * 2 / 1024**3  # 16비트 = 2바이트
memory_int4 = total_params * 0.5 / 1024**3  # 4비트 = 0.5바이트

print(f"\n[이론적 메모리 사용량 계산]")
print(f"FP32 (32-bit): {memory_fp32:.2f} GB")
print(f"FP16 (16-bit): {memory_fp16:.2f} GB")
print(f"INT4 (4-bit):  {memory_int4:.2f} GB")

print(f"\n[메모리 절감 효과]")
print(f"FP32 → FP16: {(1 - memory_fp16/memory_fp32)*100:.0f}% 절감")
print(f"FP16 → INT4: {(1 - memory_int4/memory_fp16)*100:.0f}% 절감")
print(f"FP32 → INT4: {(1 - memory_int4/memory_fp32)*100:.0f}% 절감")

[모델 파라미터 분석]
총 파라미터 수: 1,543,714,304
학습 가능한 파라미터: 1,543,714,304

[이론적 메모리 사용량 계산]
FP32 (32-bit): 5.75 GB
FP16 (16-bit): 2.88 GB
INT4 (4-bit):  0.72 GB

[메모리 절감 효과]
FP32 → FP16: 50% 절감
FP16 → INT4: 75% 절감
FP32 → INT4: 88% 절감


In [5]:
# FP32 vs FP16 숫자 표현 비교 체험
print("=" * 50)
print("[FP32 vs FP16 숫자 표현 비교]")
print("=" * 50)

# 테스트할 값들
test_values = [
    3.141592653589793,   # 원주율 (정밀도 테스트)
    0.00001,             # 아주 작은 수
    65504.0,             # FP16 최대값 근처
    0.123456789,         # 일반적인 소수
]

for value in test_values:
    fp32 = torch.tensor(value, dtype=torch.float32)
    fp16 = torch.tensor(value, dtype=torch.float16)
    
    print(f"\n원본값: {value}")
    print(f"FP32:   {fp32.item():.15f}")
    print(f"FP16:   {fp16.item():.15f}")
    print(f"오차:   {abs(value - fp16.item()):.15f}")

print("\n" + "=" * 50)
print("💡 관찰: FP16은 정밀도가 낮아 작은 오차가 발생합니다.")
print("   이 오차가 모델 전체에 누적되면 품질 저하로 이어질 수 있습니다.")

[FP32 vs FP16 숫자 표현 비교]

원본값: 3.141592653589793
FP32:   3.141592741012573
FP16:   3.140625000000000
오차:   0.000967653589793

원본값: 1e-05
FP32:   0.000009999999747
FP16:   0.000010013580322
오차:   0.000000013580322

원본값: 65504.0
FP32:   65504.000000000000000
FP16:   65504.000000000000000
오차:   0.000000000000000

원본값: 0.123456789
FP32:   0.123456791043282
FP16:   0.123474121093750
오차:   0.000017332093750

💡 관찰: FP16은 정밀도가 낮아 작은 오차가 발생합니다.
   이 오차가 모델 전체에 누적되면 품질 저하로 이어질 수 있습니다.


In [6]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: 양자화 수식을 직접 코드로 실행하여 체험
#
# [양자화란?]
# 실수(0.5, -1.2 등)를 정수(3, -2 등)로 변환하는 과정이다.
# 변환 과정에서 "반올림"이 발생하므로 원본과 약간의 오차가 생긴다.
# 비트 수가 적을수록(INT4 < INT8) 표현 가능한 단계가 줄어 오차가 커진다.
#
# [수식]
# 양자화:   Q(x) = round(x / scale) + zero_point
# 역양자화: x' = (Q(x) - zero_point) × scale
#
# scale = 데이터 범위를 정수 범위에 매핑하는 "축척 비율"
# zero_point = 실수 0이 매핑되는 정수 값
#
# [예상 결과]
# INT8(256단계): 오차가 매우 작음 (거의 원본과 동일)
# INT4(16단계): 오차가 INT8의 약 10~20배 → 이것이 품질 저하의 원인
# ═══════════════════════════════════════════════════════════

# 1. 원본 텐서: LLM 가중치 5개를 시뮬레이션
# 실제 LLM은 수십억 개의 가중치를 갖지만, 원리는 동일하다.
original = torch.tensor([0.5, 1.2, -0.3, 2.1, -1.5], dtype=torch.float16)
print(f'1. 원본 텐서 (FP16): {original.tolist()}')

# 2. 데이터 범위 분석: 양자화의 scale을 계산하기 위한 준비
# scale = (최대값 - 최소값) / (표현 가능한 단계 수)
x_min, x_max = original.min().item(), original.max().item()
print(f'2. 데이터 범위: [{x_min:.4f}, {x_max:.4f}]')

# ========== INT8 양자화 (8비트 = 256단계) ==========
# 256단계로 나누므로 scale이 매우 작다 → 정밀한 표현 가능
levels_int8 = 2**8 - 1  # 255 (0~255까지 256개 값)
scale_int8 = (x_max - x_min) / levels_int8  # 각 단계의 간격
zero_point_int8 = round(-x_min / scale_int8) # 0이 매핑되는 정수

# ★ 핵심 수식: Q(x) = round(x / scale) + zero_point
quantized_int8 = torch.round(original / scale_int8) + zero_point_int8
# 역양자화: 정수를 다시 실수로 복원 (이때 오차가 발생한다)
dequantized_int8 = (quantized_int8 - zero_point_int8) * scale_int8

print(f'\n3. INT8 양자화 (256단계):')
print(f'   scale = {scale_int8:.6f} (각 단계의 간격)')
print(f'   양자화 값:  {quantized_int8.tolist()} ← 정수로 변환됨')
print(f'   복원 값:    {[round(v, 4) for v in dequantized_int8.tolist()]} ← 원본과 비교')
error_int8 = (original.float() - dequantized_int8.float()).abs().mean().item()
print(f'   평균 오차:  {error_int8:.6f} ← 256단계이므로 오차가 매우 작다')

# ========== INT4 양자화 (4비트 = 16단계) ==========
# 16단계로 나누므로 scale이 크다 → 거친 표현, 더 큰 오차
levels_int4 = 2**4 - 1  # 15 (0~15까지 16개 값)
scale_int4 = (x_max - x_min) / levels_int4  # INT8보다 ~17배 큰 간격
zero_point_int4 = round(-x_min / scale_int4)

quantized_int4 = torch.round(original / scale_int4) + zero_point_int4
quantized_int4 = quantized_int4.clamp(0, levels_int4)  # 0~15 범위로 제한
dequantized_int4 = (quantized_int4 - zero_point_int4) * scale_int4

print(f'\n4. INT4 양자화 (16단계):')
print(f'   scale = {scale_int4:.6f} (INT8의 {scale_int4/scale_int8:.0f}배 → 더 거친 표현)')
print(f'   양자화 값:  {quantized_int4.tolist()} ← 0~15 사이 정수')
print(f'   복원 값:    {[round(v, 4) for v in dequantized_int4.tolist()]} ← 원본과 차이 발생')
error_int4 = (original.float() - dequantized_int4.float()).abs().mean().item()
print(f'   평균 오차:  {error_int4:.6f} ← INT8보다 {error_int4/max(error_int8,1e-10):.0f}배 큰 오차')

# [결과 해석]
# INT4의 오차가 INT8보다 크지만, 이 정도는 LLM에서 허용 가능한 수준이다.
# 다만 이 "작은 오차"가 수십억 개 파라미터에 걸쳐 누적되면
# 특히 노이즈/오타 같은 비정상 입력에서 품질 저하로 이어진다. (챕터 5)
print(f'\n→ INT4 오차가 INT8보다 크지만, 단일 가중치 수준에서는 허용 범위이다.')
print(f'  문제는 이 오차가 15억 개 파라미터에 걸쳐 "누적"된다는 것이다. (챕터 5에서 확인)')

1. 원본 텐서 (FP16): [0.5, 1.2001953125, -0.300048828125, 2.099609375, -1.5]
2. 데이터 범위: [-1.5000, 2.0996]

3. INT8 양자화 (256단계):
   scale = 0.014116 (각 단계의 간격)
   양자화 값:  [141.0, 191.0, 85.0, 255.0, 0.0] ← 정수로 변환됨
   복원 값:    [0.4941, 1.2002, -0.2964, 2.1035, -1.4961] ← 원본과 비교
   평균 오차:  0.003467 ← 256단계이므로 오차가 매우 작다

4. INT4 양자화 (16단계):
   scale = 0.239974 (INT8의 17배 → 더 거친 표현)
   양자화 값:  [8.0, 11.0, 5.0, 15.0, 0.0] ← 0~15 사이 정수
   복원 값:    [0.48, 1.2002, -0.24, 2.1602, -1.4395] ← 원본과 차이 발생
   평균 오차:  0.040234 ← INT8보다 12배 큰 오차

→ INT4 오차가 INT8보다 크지만, 단일 가중치 수준에서는 허용 범위이다.
  문제는 이 오차가 15억 개 파라미터에 걸쳐 "누적"된다는 것이다. (챕터 5에서 확인)


In [8]:
# TODO: FP16 모델로 추론 수행 및 latency/memory 측정

def measure_inference(model, tokenizer, prompt, max_new_tokens=128):
    """모델 추론을 수행하고 latency와 memory를 측정합니다."""
    
    # 메모리 측정 (추론 전)
    torch.cuda.synchronize()
    memory_before = torch.cuda.memory_allocated() / 1024**3
    
    # 입력 토큰화
    messages = [
        {"role": "user", "content": prompt}
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        return_dict=True,
        add_generation_prompt=True
    )
    input_ids = inputs["input_ids"].to(model.device)
    input_length = input_ids.shape[1]
    
    # 재현성을 위한 seed 설정
    torch.manual_seed(42)
    
    # TODO: 추론 수행 및 latency/memory 측정 코드를 작성해주세요
    # 1. model.generate()로 추론 수행 (input_ids 사용)
    # 2. torch.cuda.max_memory_allocated()로 메모리 측정
    # 3. tokenizer.decode()로 응답 디코딩 (outputs[0][input_length:] 사용)

    # 추론 시간 측정
    start_time = time.time()

    with torch.no_grad():
        # TODO: 추론 수행
        outputs = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )

    torch.cuda.synchronize()
    end_time = time.time()

    # TODO: 메모리 측정 (추론 후)
    memory_after = torch.cuda.max_memory_allocated() / 1024 ** 3

    # TODO: 결과 디코딩
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    latency = end_time - start_time
    
    return response, latency, memory_after

# 테스트 프롬프트
test_prompt = "서울에서 부산까지 KTX로 얼마나 걸리나요?"

# FP16 추론 실행
response_fp16, latency_fp16, memory_fp16 = measure_inference(model_fp16, tokenizer, test_prompt)

print("=" * 50)
print("[FP16 모델 추론 결과]")
print("=" * 50)
print(f"입력: {test_prompt}")
print(f"응답: {response_fp16}")
print(f"\nLatency: {latency_fp16:.2f}초")
print(f"Memory: {memory_fp16:.2f} GB")
print("\n⚠️ 체감: 메모리를 많이 사용하네! 더 큰 모델은 16GB에 못 올리겠다.")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


[FP16 모델 추론 결과]
입력: 서울에서 부산까지 KTX로 얼마나 걸리나요?
응답: KTX는 한국의 대형 고속 철도선으로, 서울과 부산을 연결하는 빠른 교통수단입니다. 일반적인 기준으로 부산에 도착하는데 약 4시간이 걸립니다.

그러나 실시간 운행 상태와 시간 변동에 따라 이 값이 달라질 수 있으므로, 실제 운행 상황에 따라 확인하시는 것이 좋습니다. 또한, 휴일이나 특정 요소가 있을 경우 운임이 상승할 수 있으니 충분히 미리 예약하시거나 지연 사항이

Latency: 3.61초
Memory: 2.90 GB

⚠️ 체감: 메모리를 많이 사용하네! 더 큰 모델은 16GB에 못 올리겠다.


In [9]:
# FP16 결과 저장 (나중에 INT4와 비교용)
fp16_results = {
    "latency": latency_fp16,
    "memory": memory_fp16,
    "model_memory": model_memory_fp16,  # 모델 가중치 메모리
    "response": response_fp16
}

print(f"FP16 모델 가중치 메모리: {model_memory_fp16:.2f} GB")
print(f"FP16 추론 피크 메모리: {memory_fp16:.2f} GB")
print(f"16GB VRAM의 {(memory_fp16/16)*100:.1f}% 사용 중")

# FP16 모델 메모리 해제 (INT4 로딩을 위해)
del model_fp16
torch.cuda.empty_cache()
print("\nFP16 모델 메모리 해제 완료")

FP16 모델 가중치 메모리: 2.88 GB
FP16 추론 피크 메모리: 2.90 GB
16GB VRAM의 18.1% 사용 중

FP16 모델 메모리 해제 완료


## INT4

In [15]:
# TODO: INT4 양자화 모델 로딩 및 FP16과 비교

# TODO: BitsAndBytesConfig로 INT4 양자화 설정을 작성해주세요
# 1. BitsAndBytesConfig로 INT4 양자화 설정 생성
#    - load_in_4bit=True
#    - bnb_4bit_compute_dtype=torch.float16
#    - bnb_4bit_quant_type="nf4"
#    - bnb_4bit_use_double_quant=True

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)  # TODO: BitsAndBytesConfig 설정

# INT4 모델 로딩
print("INT4 양자화 모델 로딩 중...")
torch.cuda.reset_peak_memory_stats()  # 메모리 통계 초기화

# TODO: AutoModelForCausalLM.from_pretrained()로 INT4 모델을 로딩해주세요
# AutoModelForCausalLM.from_pretrained()에 quantization_config 전달

model_int4 = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map='auto',
)  # TODO: INT4 모델 로딩

# 모델 로딩 직후 메모리 측정 (가중치 메모리)
model_memory_int4 = torch.cuda.memory_allocated() / 1024**3
print(f"INT4 모델 로딩 완료: {model_name}")
print(f"INT4 모델 메모리: {model_memory_int4:.2f} GB")

INT4 양자화 모델 로딩 중...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

INT4 모델 로딩 완료: Qwen/Qwen2.5-1.5B-Instruct
INT4 모델 메모리: 1.75 GB


In [16]:
# INT4 추론 실행
response_int4, latency_int4, memory_int4 = measure_inference(model_int4, tokenizer, test_prompt)

print("=" * 50)
print("[INT4 모델 추론 결과]")
print("=" * 50)
print(f"입력: {test_prompt}")
print(f"응답: {response_int4}")
print(f"\nLatency: {latency_int4:.2f}초")
print(f"추론 피크 메모리: {memory_int4:.2f} GB")

[INT4 모델 추론 결과]
입력: 서울에서 부산까지 KTX로 얼마나 걸리나요?
응답: KTX(Korean Train System)는 한국의 철도 서비스를 위한 고속철도입니다. 서울에서 부산으로 KTX를 타면 일반적인 시간은 약 3시간 40분 정도가 걸립니다. 하지만 지정된 경로와 운행 시각에 따라 다소 변동이 발생할 수 있습니다. 따라서 더욱 정확한 시간을 알고 싶으시다면 실시간 정보를 확인하는 것이 좋습니다.

Latency: 3.25초
추론 피크 메모리: 4.47 GB


In [17]:
# FP16 vs INT4 비교
print("\n" + "=" * 70)
print("[FP16 vs INT4 비교]")
print("=" * 70)
print(f"{'지표':<20} {'FP16':>15} {'INT4':>15} {'변화율':>15}")
print("-" * 70)

# 모델 가중치 메모리 비교 (핵심 지표)
model_memory_change = ((model_memory_int4 - fp16_results['model_memory']) / fp16_results['model_memory']) * 100
print(f"{'모델 메모리 (GB)':<20} {fp16_results['model_memory']:>15.2f} {model_memory_int4:>15.2f} {model_memory_change:>14.1f}%")

# Latency 비교
latency_change = ((latency_int4 - fp16_results['latency']) / fp16_results['latency']) * 100
print(f"{'Latency (초)':<20} {fp16_results['latency']:>15.2f} {latency_int4:>15.2f} {latency_change:>14.1f}%")

# 추론 피크 메모리 비교
peak_memory_change = ((memory_int4 - fp16_results['memory']) / fp16_results['memory']) * 100
print(f"{'피크 메모리 (GB)':<20} {fp16_results['memory']:>15.2f} {memory_int4:>15.2f} {peak_memory_change:>14.1f}%")

print("\n✅ 체감: INT4 양자화로 모델 메모리가 대폭 줄었다!")
print("   → 같은 GPU에서 더 큰 모델을 로딩할 수 있게 됨")


[FP16 vs INT4 비교]
지표                              FP16            INT4             변화율
----------------------------------------------------------------------
모델 메모리 (GB)                     2.88            1.75          -39.1%
Latency (초)                     3.61            3.25          -10.0%
피크 메모리 (GB)                     2.90            4.47           54.5%

✅ 체감: INT4 양자화로 모델 메모리가 대폭 줄었다!
   → 같은 GPU에서 더 큰 모델을 로딩할 수 있게 됨


## TTP

In [18]:
# 환경 변화 입력 샘플 정의
test_samples = [
    {
        "type": "정상",
        "prompt": "서울에서 부산까지 KTX로 얼마나 걸리나요?",
        "expected_quality": "높음"
    },
    {
        "type": "오타",
        "prompt": "서울에셔 부산까지 KTX로 얼마나 걸리나요?",
        "expected_quality": "중간"
    },
    {
        "type": "노이즈",
        "prompt": "서울에서 부산까지... 음... KTX로 얼마나 걸리나요? 대략?",
        "expected_quality": "중간"
    },
    {
        "type": "모호함",
        "prompt": "그거 얼마나 걸려?",
        "expected_quality": "낮음"
    },
    {
        "type": "조건변화",
        "prompt": "밤 11시에 출발하면 다음날 아침에 도착할 수 있나요?",
        "expected_quality": "중간"
    }
]

print(f"총 {len(test_samples)}개의 테스트 샘플 정의 완료")

총 5개의 테스트 샘플 정의 완료


In [19]:
# 환경 변화 입력 테스트
env_change_results = []

print("=" * 70)
print("[INT4 모델 - 환경 변화 입력 테스트]")
print("=" * 70)

for sample in test_samples:
    response, latency, _ = measure_inference(model_int4, tokenizer, sample["prompt"], max_new_tokens=100)
    
    env_change_results.append({
        "type": sample["type"],
        "prompt": sample["prompt"],
        "response": response,
        "latency": latency
    })
    
    print(f"\n[{sample['type']}]")
    print(f"입력: {sample['prompt']}")
    print(f"응답: {response[:200]}..." if len(response) > 200 else f"응답: {response}")
    print("-" * 70)

[INT4 모델 - 환경 변화 입력 테스트]

[정상]
입력: 서울에서 부산까지 KTX로 얼마나 걸리나요?
응답: KTX(Korean Train System)는 한국의 철도 서비스를 위한 고속철도입니다. 서울에서 부산으로 KTX를 타면 일반적인 시간은 약 3시간 40분 정도가 걸립니다. 하지만 지정된 경로와 운행 시각에 따라 다소 변동이 발생할 수 있습니다. 따라서 더욱 정확한 시간을 알고 싶으시다면 실시간 정보를 확인하는 것이 좋습니다.
----------------------------------------------------------------------

[오타]
입력: 서울에셔 부산까지 KTX로 얼마나 걸리나요?
응답: KTX(Korean Train System)는 한국의 철도 서비스를 위한 고속철도입니다. 서울에서 부산으로 KTX를 타고 가면 다음과 같은 시간을 기대할 수 있습니다:

1. 2시간 30분 - 경부선(서해선)
   (경상남도 - 경남북부권)

2. 2시간 45분 - 경전선(동부선)
   (경기북부
----------------------------------------------------------------------

[노이즈]
입력: 서울에서 부산까지... 음... KTX로 얼마나 걸리나요? 대략?
응답: KTX(Korean Train System)는 한국의 철도 서비스를 위한 주요 시스템 중 하나입니다. 서울과 부산을 연결하는 대형 열차인 A10와 S10 등 다양한 종류의 열차가 운영됩니다.

물론에 대해 알려드리면 좋겠어요! 하지만 제가 알고 있는 정보를 바탕으로 대략적인 시간을 예측해 드릴 수 있습니다:

- A10은 서울에서 부산까지 �
----------------------------------------------------------------------

[모호함]
입력: 그거 얼마나 걸려?
응답: 죄송합니다, 저는 자연어 처리 모델이기 때문에 영상이나 이미지, 음성 같은 다중 형식의 입력을 이해하거나 처리할

In [21]:
# TTP 템플릿 정의

# TTP-A: 출력 형식/제약 강화
TTP_A_TEMPLATE = """다음 질문에 답해주세요. 

**규칙:**
1. 오타나 불명확한 표현이 있어도 의도를 파악하세요.
2. 교통 관련 질문은 대략적인 소요 시간(시간 단위)으로 답하세요.
3. 맥락이 부족하면 일반적인 상황을 가정하여 답하세요.
4. 간결하게 핵심만 답하세요.

**질문:** {question}

**답변:**"""

# TTP-B: few-shot 예시 포함
TTP_B_TEMPLATE = """다음은 교통 관련 질문과 답변의 예시입니다.

**예시:**
Q: 서울역에서 대전역까지 KTX로 얼마나 걸려요?
A: 서울역에서 대전역까지 KTX로 약 50분~1시간 정도 소요됩니다.

이제 아래 질문에 동일한 형식으로 답해주세요.

**질문:** {question}

**답변:**"""

print("TTP 템플릿 정의 완료:")
print("- TTP-A: 출력 형식/제약 강화")
print("- TTP-B: few-shot 예시 포함")

TTP 템플릿 정의 완료:
- TTP-A: 출력 형식/제약 강화
- TTP-B: few-shot 예시 포함


In [22]:
# TODO: TTP 템플릿 적용 후 품질 변화 비교

def apply_ttp(template, question):
    """TTP 템플릿을 적용하여 프롬프트를 생성합니다."""
    # TODO: template.format()을 사용하여 TTP 적용 함수를 작성해주세요
    # template.format()을 사용하여 {question}을 실제 질문으로 대체
    return template.format(question=question)  # TODO: 구현하세요

# TTP 적용 결과 저장
ttp_results = []

print("=" * 80)
print("[TTP 적용 테스트]")
print("=" * 80)

for sample in test_samples:
    original_prompt = sample["prompt"]
    
    # TODO: TTP-A와 TTP-B 템플릿을 적용하여 추론을 수행해주세요
    # 1. TTP-A 템플릿 적용하여 프롬프트 생성
    # 2. TTP-B 템플릿 적용하여 프롬프트 생성

    # TTP-A 적용
    ttp_a_prompt = apply_ttp(TTP_A_TEMPLATE, original_prompt)
    response_a, _, _ = measure_inference(model_int4, tokenizer, ttp_a_prompt, max_new_tokens=100)

    # TTP-B 적용
    ttp_b_prompt = apply_ttp(TTP_B_TEMPLATE, original_prompt)
    response_b, _, _ = measure_inference(model_int4, tokenizer, ttp_b_prompt, max_new_tokens=100)

    ttp_results.append({
        "type": sample["type"],
        "original": original_prompt,
        "no_ttp": env_change_results[[r["type"] for r in env_change_results].index(sample["type"])]["response"],
        "ttp_a": response_a,
        "ttp_b": response_b
    })

    print(f"\n[{sample['type']}]")
    print(f"원본: {original_prompt}")
    print(f"TTP-A 응답: {response_a[:150]}..." if len(response_a) > 150 else f"TTP-A 응답: {response_a}")
    print(f"TTP-B 응답: {response_b[:150]}..." if len(response_b) > 150 else f"TTP-B 응답: {response_b}")
    print("-" * 80)

[TTP 적용 테스트]

[정상]
원본: 서울에서 부산까지 KTX로 얼마나 걸리나요?
TTP-A 응답: KTX(Korean Train System)는 매우 빠른 편차를 제공하며, 일반적으로 서울과 부산 사이의 거리를 3시간 내외가 걸릴 것으로 예상됩니다. 하지만 실적에는 차이가 있을 수 있으므로 실제 탑승 시 기간에 따라 달라질 수 있습니다.
TTP-B 응답: 서울에서 부산까지 KTX로 약 7시간~8시간 정도 소요됩니다.
--------------------------------------------------------------------------------

[오타]
원본: 서울에셔 부산까지 KTX로 얼마나 걸리나요?
TTP-A 응답: KTX(Korean Train System)는 현재의 기술과 투자로 지속적으로 발전하고 있습니다. 따라서 현재 최신 정보를 바탕으로 다음과 같은 추정 시간을 제공할 수 있습니다:

- **상하행 모두**: 6시간 25분

여기서 "상하행"은 상하역 방식의 열차입니다. ...
TTP-B 응답: 서울에시 부산까지 KTX로 약 2시간~2 시간 30 분 정도 소요됩니다.
--------------------------------------------------------------------------------

[노이즈]
원본: 서울에서 부산까지... 음... KTX로 얼마나 걸리나요? 대략?
TTP-A 응답: KTX(Korean Train System)는 7시간 5분이 걸립니다.
TTP-B 응답: 서울에서 부산까지 KTX로 약 3시간~4시간 정도 소요됩니다.
--------------------------------------------------------------------------------

[모호함]
원본: 그거 얼마나 걸려?
TTP-A 응답: 죄송합니다, "그거"의 정확한 내용이나 상황을 알려주셔야 더 정확한 답변을 드릴 수 있습니다. 다른 정보가 있으면 말씀해주시면 감사하겠습니다.
TTP-B 응답: 물론입니